# Week 5 Exercise Solution - Dynamic URL RAG System

In [ ]:
# Imports
# Standard library
import os
import re
from pathlib import Path

# Environment
from dotenv import load_dotenv

# Scraping
from bs4 import BeautifulSoup

# LangChain
from langchain_community.document_loaders import RecursiveUrlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

# Advanced (LLM chunking, query rewriting, reranking)
from openai import OpenAI
from chromadb import PersistentClient
from pydantic import BaseModel, Field
from tenacity import retry, wait_exponential
from tqdm import tqdm
from multiprocessing import Pool

# UI
import gradio as gr

In [ ]:
# Load environment
load_dotenv(override=True)
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")

In [ ]:
# Constant 
DB_NAME = "dynamic_vector_db"
EMBED_MODEL= "text-embedding-3-large"
LLM_MODEL = "gpt-4.1-nano"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
RETRIEVAL_K = 10
RETRIEVAL_K_RERANK = 20
WORKERS = 3

# Global state
vectorstore = None
current_url = ""
vectorstore_state = ""

In [ ]:
# Connect to client
openrouter_url = "https://openrouter.ai/api/v1"

def get_client():
    return OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


def _pydantic_to_json_schema(pydantic_model: type[BaseModel]) -> dict:
    """Build OpenAI response_format from a Pydantic model for structured output."""
    schema = pydantic_model.model_json_schema()
    # OpenAI expects "name" and "strict"; schema can include $defs from Pydantic
    return {
        "type": "json_schema",
        "json_schema": {
            "name": pydantic_model.__name__,
            "strict": True,
            "schema": schema,
        },
    }


def _completion(messages: list, model: str = None, json_mode: bool = False, response_format=None):
    """
    OpenAI-compatible completion. Supports structured output via Pydantic.
    - json_mode: use response_format={"type": "json_object"}
    - response_format: a Pydantic BaseModel class (e.g. ChunksSchema, RankOrder)
      to get strict structured output
    """
    client = get_client()
    kwargs = {"model": model or LLM_MODEL, "messages": messages}
    if response_format is not None and isinstance(response_format, type) and issubclass(response_format, BaseModel):
        kwargs["response_format"] = _pydantic_to_json_schema(response_format)
    elif json_mode:
        kwargs["response_format"] = {"type": "json_object"}
    return client.chat.completions.create(**kwargs)

## Scraping

In [ ]:
import re
import time
import requests
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from langchain_core.documents import Document

# Optional: Selenium (install selenium, webdriver-manager)
try:
    from selenium import webdriver
    from selenium.webdriver.chrome.options import Options
    from selenium.webdriver.chrome.service import Service
    from webdriver_manager.chrome import ChromeDriverManager
    SELENIUM_AVAILABLE = True
except ImportError:
    SELENIUM_AVAILABLE = False

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# Thresholds for JS detection
MIN_STATIC_CONTENT_CHARS = 300
SELENIUM_WAIT_SECONDS = 3


def _get_body_text(html: str) -> str:
    """Extract body text only (for JS detection)."""
    soup = BeautifulSoup(html, "html.parser")
    for tag in soup(["script", "style", "noscript"]):
        tag.decompose()
    if soup.body:
        return soup.body.get_text(separator=" ", strip=True)
    return ""


def _has_spa_indicators(html: str) -> bool:
    """Check for common SPA/JS framework indicators in raw HTML."""
    lower = html.lower()
    patterns = [
        'id="root"',
        'id="app"',
        'id=\'root\'',
        'id=\'app\'',
        "__next_data__",
        "data-reactroot",
        "ng-version",
        "v-cloak",
        "vue-app",
        "react-root",
    ]
    return any(p in lower for p in patterns)


def is_js_rendered(html: str, url: str = "") -> bool:
    """
    Heuristic: page is likely JS-rendered if:
    1. Body text is very short (< MIN_STATIC_CONTENT_CHARS)
    2. AND (has SPA indicators OR body is mostly empty)
    """
    body_text = _get_body_text(html)
    text_len = len(body_text)

    if text_len >= MIN_STATIC_CONTENT_CHARS:
        return False

    if _has_spa_indicators(html):
        return True

    # Very little content and no clear structure
    if text_len < 100:
        return True

    return False


def extract_headers_and_content(html: str) -> tuple[list[str], str]:
    """
    Extract (list of headers), (body text).
    Headers: ["H1: Title", "H2: Section", ...]
    Body: plain text, script/style removed.
    """
    soup = BeautifulSoup(html, "html.parser")

    headers_list = []
    for tag in soup.find_all(["h1", "h2", "h3", "h4", "h5", "h6"]):
        text = tag.get_text(strip=True)
        if text:
            headers_list.append(f"{tag.name.upper()}: {text}")

    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()

    body = soup.get_text(separator="\n", strip=True)
    body = re.sub(r"\n\n+", "\n\n", body).strip()

    return headers_list, body


def page_to_document(url: str, html: str, title: str = "") -> Document:
    """
    Create a LangChain Document with headers and content as separate fields.
    page_content combines both for embedding/retrieval.
    metadata holds source, title, headers, body.
    """
    headers_list, body = extract_headers_and_content(html)
    if not title:
        soup = BeautifulSoup(html, "html.parser")
        title = soup.title.string if soup.title and soup.title.string else "No title"

    page_content = "Headers:\n" + "\n".join(headers_list) + "\n\nContent:\n" + body

    return Document(
        page_content=page_content,
        metadata={
            "source": url,
            "title": title,
            "headers": "".join(headers_list),
            "body": body,
        },
    )


def fetch_html_requests(url: str) -> str:
    """Fetch raw HTML via requests (no JS execution)."""
    resp = requests.get(url, headers=HEADERS, timeout=15)
    resp.raise_for_status()
    return resp.text


def fetch_html_selenium(url: str) -> str:
    """Fetch rendered HTML via Selenium (JS executed)."""
    if not SELENIUM_AVAILABLE:
        raise RuntimeError("Selenium not installed. pip install selenium webdriver-manager")
    options = Options()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")

    service = Service(ChromeDriverManager().install())
    driver = webdriver.Chrome(service=service, options=options)
    try:
        driver.get(url)
        time.sleep(SELENIUM_WAIT_SECONDS)
        return driver.page_source
    finally:
        driver.quit()


def load_page(url: str, force_selenium: bool = False) -> Document:
    """
    Load a single page into a Document.
    - Tries requests first.
    - If is_js_rendered(html), falls back to Selenium.
    - Uses headers + content extraction and page_to_document.
    """
    if force_selenium:
        html = fetch_html_selenium(url)
    else:
        html = fetch_html_requests(url)
        if is_js_rendered(html, url):
            html = fetch_html_selenium(url)

    soup = BeautifulSoup(html, "html.parser")
    title = soup.title.string if soup.title and soup.title.string else ""

    return page_to_document(url, html, title)


def load_pages_from_url(
    url: str,
    follow_links: bool = False,
    max_pages: int = 5,
    force_selenium: bool = False,
) -> list[Document]:
    """
    Load the main page and optionally follow links.
    JS check and Selenium fallback apply to each page.
    """
    docs = [load_page(url, force_selenium=force_selenium)]

    if not follow_links:
        return docs

    try:
        html = fetch_html_requests(url)
        if is_js_rendered(html, url) and SELENIUM_AVAILABLE:
            html = fetch_html_selenium(url)
    except Exception:
        return docs

    soup = BeautifulSoup(html, "html.parser")
    seen = {url}

    for a in soup.find_all("a", href=True):
        href = a["href"]
        full = urljoin(url, href) if not href.startswith("http") else href
        if not full.startswith("http") or full in seen:
            continue
        seen.add(full)
        if len(docs) >= max_pages:
            break
        try:
            docs.append(load_page(full, force_selenium=force_selenium))
        except Exception:
            pass

    return docs

## Chunking

In [ ]:
# Pydantic schemas for LLM chunking (add to notebook)

class ChunkSchema(BaseModel):
    headline: str = Field(
        description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query"
    )
    summary: str = Field(
        description="A few sentences summarizing the content of this chunk to answer common questions"
    )
    original_text: str = Field(
        description="The original text of this chunk from the provided document, exactly as is, not changed in any way"
    )


class ChunksSchema(BaseModel):
    chunks: list[ChunkSchema]


AVERAGE_CHUNK_SIZE = 200  # Target ~200 chars per chunk for LLM to decide boundaries


def _make_llm_chunk_prompt(doc: Document) -> str:
    """Build prompt for LLM to split a scraped page into semantic chunks."""
    text = doc.page_content
    source = doc.metadata.get("source", "unknown")
    title = doc.metadata.get("title", "Unknown")
    how_many = max(1, (len(text) // AVERAGE_CHUNK_SIZE) + 1)

    
    prompt = f"""
    You take a document from a web page and split it into overlapping chunks for a RAG knowledge base.

    The document is from: {source}
    Page title: {title}

    A chatbot will use these chunks to answer questions about the page content.
    You should divide the document as you see fit, ensuring the entire document is covered across the chunks - don't leave anything out.
    This document should probably be split into at least {how_many} chunks, but you can have more or less as appropriate.
    There should be overlap between chunks (typically ~25% or ~50 words) so the same context appears in multiple chunks for better retrieval.

    For each chunk provide:
    1. headline: a brief heading likely to match user queries
    2. summary: a few sentences summarizing the chunk for common questions
    3. original_text: the exact text of the chunk, unchanged

    Together your chunks must represent the entire document with overlap.

    Document:

    {text}

    Respond with the chunks in the required JSON format.
    """

    return prompt


@retry(wait=wait_exponential(multiplier=1, min=2, max=60))
def _process_document_llm(doc: Document, model: str, client) -> list[Document]:
    """Process a single document with LLM chunking. Returns list of LangChain Documents."""
    
    messages = [{"role": "user", "content": _make_llm_chunk_prompt(doc)}]
    response = _completion(messages, model=model, response_format=ChunksSchema)
    reply = response.choices[0].message.content
    parsed = ChunksSchema.model_validate_json(reply)

    results = []
    base_metadata = {
        "source": doc.metadata.get("source", ""),
        "title": doc.metadata.get("title", ""),
        "type": "webpage",
    }
    for chunk in parsed.chunks:
        page_content = f"{chunk.headline}\n\n{chunk.summary}\n\n{chunk.original_text}"
        results.append(
            Document(page_content=page_content, metadata=base_metadata.copy())
        )
        
    return results


def create_chunks_llm(documents: list[Document], model: str = None, client=None) -> list[Document]:
    """
    Create chunks using LLM-based semantic splitting.
    Used when "Use LLM chunking" checkbox is enabled.
    """
    model = model or LLM_MODEL
    client = client or get_client()
    
    all_chunks = []
    with Pool(processes=WORKERS) as pool:
        for doc in tqdm(pool.imap_unordered(_process_document_llm, [documents, model, client]), desc="LLM chunking", total=len(documents)):
            # chunks = _process_document_llm(doc, model, client)
            all_chunks.extend(doc)
    return all_chunks

def create_chunks(
    documents: list[Document],
    use_llm_chunking: bool = False,
) -> list[Document]:
    """Create chunks. Uses LLM when use_llm_chunking=True, else RecursiveCharacterTextSplitter."""
    if use_llm_chunking:
        return create_chunks_llm(documents)
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )
    return splitter.split_documents(documents)

## Ingestion

In [ ]:
def ingest_from_url(url: str, use_llm_chunking: bool = False) -> str:
    global vectorstore, current_url

    docs = load_pages_from_url(url)
    
    if not docs:
        return f"Error: No content loaded from {url}"

    chunks = create_chunks(docs, use_llm_chunking)
    
    if not chunks:
        return f"Error: No chunks created from {url}"

    embeddings = OpenAIEmbeddings(model=EMBED_MODEL, base_url=openrouter_url, api_key=openrouter_api_key)

    # Clear existing Chroma before creating new
    if os.path.exists(DB_NAME):
        existing = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
        existing.delete_collection()

    # Create new vectorstore from chunks
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=DB_NAME,
    )
    # vectorstore.persist()

    current_url = url
    return f"Indexed {len(chunks)} chunks from {url}"

## Retrieval

In [ ]:
# RankOrder schema for reranking
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="Order of relevance (1-based chunk ids), most relevant first"
    )


def rewrite_query(question: str, history: list = None) -> str:
    """Rewrite question into a focused search query for the knowledge base."""
    history = history or []
    history_text = "\n".join(
        f"User: {m.get('content', m.get('message', ''))}"
        if isinstance(m, dict) else str(m)
        for m in history[-4:]
    )
    prompt = f"""
    You are about to search a knowledge base built from web pages.
    Create a short, focused query that will surface relevant content.
    Use only the rewritten query, nothing else.

    Conversation history:
    {history_text}

    Current question: {question}

    Rewrite as a concise search query:
    """
    response = _completion([{"role": "user", "content": prompt}])
    return response.choices[0].message.content.strip()


def rerank_chunks(question: str, chunks: list[Document]) -> list[Document]:
    """Reorder chunks by relevance to the question using LLM."""
    if len(chunks) <= 1:
        return chunks
    system = """You are a document re-ranker. Order the chunks by relevance to the question.
Reply ONLY with a JSON object: {"order": [1, 2, 3, ...]} where numbers are 1-based chunk ids."""
    user = f"Question: {question}\n\nChunks:\n"
    for i, c in enumerate(chunks):
        user += f"# CHUNK ID {i+1}:\n{c.page_content[:500]}...\n\n"
    user += "Reply ONLY with JSON: {\"order\": [ ... ]}"
    response = _completion(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        response_format=RankOrder,
    )
    order = RankOrder.model_validate_json(response.choices[0].message.content).order
    return [chunks[i - 1] for i in order if 1 <= i <= len(chunks)]


def fetch_context(
    question: str,
    history: list = None,
    use_rewrite: bool = False,
    use_rerank: bool = False,
) -> list[Document]:
    """
    Retrieve relevant chunks for the question.
    - use_rewrite: rewrite question before search, merge results
    - use_rerank: retrieve more, rerank, then take top RETRIEVAL_K
    """
    if vectorstore is None:
        return []

    history = history or []
    k = RETRIEVAL_K_RERANK if use_rerank else RETRIEVAL_K

    if use_rewrite:
        rewritten = rewrite_query(question, history)
        chunks1 = vectorstore.similarity_search(question, k=k)
        chunks2 = vectorstore.similarity_search(rewritten, k=k)
        # chunks2 (rewritten query) has priority - put first, then fill with chunks1
        seen_content = set()
        merged = []
        for c in chunks2:
            if c.page_content not in seen_content:
                seen_content.add(c.page_content)
                merged.append(c)
        for c in chunks1:
            if c.page_content not in seen_content:
                seen_content.add(c.page_content)
                merged.append(c)
        chunks = merged[:k]
    else:
        chunks = vectorstore.similarity_search(question, k=k)

    if use_rerank:
        chunks = rerank_chunks(question, chunks)

    return chunks[:RETRIEVAL_K]

## Answer Question

In [ ]:
# RAG system prompt
SYSTEM_PROMPT = """You are a helpful assistant. Answer the user's question using only the provided context from web pages they loaded.
If the context does not contain relevant information, say so. Do not make up facts.
When possible, cite the source (URL or page title).
Context:
{context}
"""

# LLM for answering (OpenRouter)
llm = ChatOpenAI(
    temperature=0,
    model=LLM_MODEL,
    base_url=openrouter_url,
    api_key=openrouter_api_key,
)


def _history_to_messages(history: list) -> list:
    """Convert Gradio chat history to message dicts for fetch_context."""
    if not history:
        return []
    # Already in messages format
    if isinstance(history[0], dict) and "role" in history[0]:
        return [{"role": m["role"], "content": m.get("content", "") or ""} for m in history]
    # Legacy tuples format (user, bot)
    msgs = []
    for user_msg, bot_msg in history:
        if user_msg:
            msgs.append({"role": "user", "content": user_msg})
        if bot_msg:
            msgs.append({"role": "assistant", "content": bot_msg})
    return msgs


def format_sources(chunks: list) -> str:
    """Format chunk references with metadata (source, title) for display."""
    if not chunks:
        return "_No sources retrieved._"
    lines = []
    for i, doc in enumerate(chunks, 1):
        source = doc.metadata.get("source", "Unknown")
        title = doc.metadata.get("title", "No title")
        preview = doc.page_content[:200].replace("\n", " ") + "..." if len(doc.page_content) > 200 else doc.page_content
        lines.append(f"**{i}. [{title}]({source})**\n   {preview}")
    return "\n\n".join(lines)


def answer_question(
    question: str,
    history: list,
    use_rewrite: bool,
    use_rerank: bool,
) -> tuple[str, str, list]:
    """
    Answer question using RAG. Returns (answer, sources_markdown, source_chunks).
    """
    if vectorstore is None:
        return "Please load a URL first to build the RAG system.", "_No sources._", []

    msg_history = _history_to_messages(history)
    docs = fetch_context(question, history=msg_history, use_rewrite=use_rewrite, use_rerank=use_rerank)

    if not docs:
        return "I couldn't find relevant context. Try rephrasing your question.", format_sources([]), []

    context = "\n\n---\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    for m in msg_history:
        if m["role"] == "user":
            messages.append(HumanMessage(content=m["content"]))
        else:
            messages.append(AIMessage(content=m["content"]))
    messages.append(HumanMessage(content=question))

    response = llm.invoke(messages)
    return response.content, format_sources(docs), docs


def answer_question_stream(
    question: str,
    history: list,
    use_rewrite: bool,
    use_rerank: bool,
):
    """
    Generator that streams the RAG answer token-by-token.
    Yields (partial_answer, sources_markdown).
    """
    if vectorstore is None:
        yield "Please load a URL first to build the RAG system.", "_No sources._"
        return

    msg_history = _history_to_messages(history)
    docs = fetch_context(question, history=msg_history, use_rewrite=use_rewrite, use_rerank=use_rerank)

    if not docs:
        yield "I couldn't find relevant context. Try rephrasing your question.", format_sources([])
        return

    context = "\n\n---\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    messages = [SystemMessage(content=system_prompt)]
    for m in msg_history:
        if m["role"] == "user":
            messages.append(HumanMessage(content=m["content"]))
        else:
            messages.append(AIMessage(content=m["content"]))
    messages.append(HumanMessage(content=question))

    sources_md = format_sources(docs)
    full_answer = ""
    for chunk in llm.stream(messages):
        full_answer += chunk.content or ""
        yield full_answer, sources_md


## Gradio UI

In [ ]:
custom_css = """
/* Modern theme with indigo/teal accents */
.gradio-container {
    font-family: 'Inter', 'SF Pro Display', system-ui, sans-serif !important;
    --primary-50: #eef2ff !important;
    --primary-100: #e0e7ff !important;
    --primary-500: #6366f1 !important;
    --primary-600: #4f46e5 !important;
    --primary-700: #4338ca !important;
    --teal-500: #14b8a6 !important;
    --teal-600: #0d9488 !important;
}
.header-box {
    text-align: center;
    padding: 2rem;
    background: linear-gradient(135deg, #4338ca 0%, #6366f1 50%, #06b6d4 100%);
    border-radius: 16px;
    color: white !important;
    font-weight: 600;
    box-shadow: 0 4px 20px rgba(99, 102, 241, 0.3);
}
.gr-button-primary {
    background: linear-gradient(135deg, #4f46e5, #06b6d4) !important;
    border: none !important;
    font-weight: 500 !important;
}
.gr-button-primary:hover {
    background: linear-gradient(135deg, #4338ca, #0d9488) !important;
    box-shadow: 0 4px 12px rgba(79, 70, 229, 0.4) !important;
}
.gr-input, .gr-textbox {
    border-radius: 10px !important;
    border: 1px solid #c7d2fe !important;
}
.gr-input:focus, .gr-textbox:focus-within {
    border-color: #6366f1 !important;
    box-shadow: 0 0 0 2px rgba(99, 102, 241, 0.2) !important;
}
.gr-form {
    border-radius: 12px !important;
}
"""


def build_rag_with_progress(url: str, use_llm_chunking: bool, progress=gr.Progress()) -> tuple[str, bool]:
    """Build RAG from URL with progress updates."""
    if not url or not url.strip().startswith("http"):
        return "Please enter a valid URL (e.g. https://en.wikipedia.org/wiki/...)", False
    try:
        progress(0.2, desc="Building RAG pipeline...")
        result = ingest_from_url(url.strip(), use_llm_chunking)
        progress(1.0, desc="Done!")
        return result, True
    except Exception as e:
        return f"Error: {str(e)}", False


def chat_fn(message, history, use_rewrite, use_rerank):
    """Chat handler: streams response token-by-token, yields (history, sources_markdown)."""
    if not message or not message.strip():
        return history, ""
    msg = message.strip()
    # history from Gradio is list of {"role": "user"|"assistant", "content": "..."}
    new_history = history + [
        {"role": "user", "content": msg},
        {"role": "assistant", "content": ""}
    ]
    sources_md = ""
    for partial, sources_md in answer_question_stream(msg, history, use_rewrite, use_rerank):
        new_history[-1]["content"] = partial
        yield new_history, sources_md


with gr.Blocks(css=custom_css, theme=gr.themes.Soft(primary_hue="indigo")) as demo:
    gr.Markdown("# Dynamic URL RAG System", elem_classes=["header-box"])

    rag_ready = gr.State(value=False)

    # === Top URL bar (visible when RAG ready) ===
    with gr.Row(visible=False) as top_url_row:
        with gr.Column(scale=4):
            url_input_top = gr.Textbox(
                label="Load new URL",
                placeholder="https://en.wikipedia.org/wiki/...",
                show_label=True,
            )
        # with gr.Column(scale=1):
            # use_llm_chunk_top = gr.Checkbox(label="LLM chunking", value=False)
        with gr.Column(scale=1):
            build_btn_top = gr.Button("Build RAG", variant="primary")

    # === Initial centered URL input (visible when RAG not ready) ===
    with gr.Row(visible=True) as initial_url_row:
        with gr.Column(scale=1, min_width=200):
            pass
        with gr.Column(scale=3, min_width=400):
            url_input = gr.Textbox(
                label="Enter a URL to build your RAG knowledge base",
                placeholder="https://en.wikipedia.org/wiki/Python_(programming_language)",
                lines=1,
                show_label=True,
            )
            with gr.Row():
                # use_llm_chunk = gr.Checkbox(label="Use LLM chunking", value=False)
                build_btn = gr.Button("Build RAG System", variant="primary", scale=2)
        with gr.Column(scale=1, min_width=200):
            pass

    status_text = gr.Markdown("", elem_id="status")

    # === Chat section (hidden until RAG ready); status also appears here ===
    with gr.Row(visible=False) as chat_row:
        with gr.Column(scale=2):
            chatbot = gr.Chatbot(label="Chat", height=500, type="messages")
            with gr.Row():
                use_rewrite = gr.Checkbox(label="Use query rewriting", value=False)
                use_rerank = gr.Checkbox(label="Use reranking", value=False)
            msg_input = gr.Textbox(
                placeholder="Ask a question about the loaded page...",
                show_label=False,
                container=False,
            )
            msg_btn = gr.Button("Send", variant="primary")
        with gr.Column(scale=1):
            gr.Markdown("### Chunk references & sources")
            sources_output = gr.Markdown("_Sources will appear here after you ask a question._")

    # --- Build RAG (initial button) ---
    def on_build(url, progress=gr.Progress()):
      # First yield: hide chat & sources, show loading text
        yield (
            gr.update(visible=True),   # initial_url_row stays
            gr.update(visible=False),  # chat_row — HIDE
            gr.update(visible=False),  # top_url_row
            gr.update(value=False),
            gr.update(value="**Building RAG pipeline...** Please wait."),
        )
        status, ok = build_rag_with_progress(url, False, progress)  # pass use_llm_chunking=False
        yield (
            gr.update(visible=not ok),
            gr.update(visible=ok),
            gr.update(visible=ok),
            gr.update(value=ok),
            gr.update(value=status),
        )

    build_btn.click(
        fn=on_build,
        inputs=[url_input],
        outputs=[initial_url_row, chat_row, top_url_row, rag_ready, status_text],
    )

    # --- Build RAG (top bar, when loading new URL) ---
    def on_build_top(url, progress=gr.Progress()):
      # First yield: hide chat & sources
        yield (
            gr.update(visible=True),
            gr.update(visible=False),  # chat_row — HIDE
            gr.update(visible=False),
            gr.update(value=False),
            gr.update(value="**Building RAG pipeline...** Please wait."),
        )
        status, ok = build_rag_with_progress(url, False, progress)
        yield (
            gr.update(visible=not ok),
            gr.update(visible=ok),
            gr.update(visible=ok),
            gr.update(value=ok),
            gr.update(value=status),
        )


    build_btn_top.click(
        fn=on_build_top,
        inputs=[url_input_top],
        outputs=[initial_url_row, chat_row, top_url_row, rag_ready, status_text],
    )

    # --- Chat ---
    def respond(message, history, rw, rk):
        yield from chat_fn(message, history, rw, rk)

    msg_btn.click(
        fn=respond,
        inputs=[msg_input, chatbot, use_rewrite, use_rerank],
        outputs=[chatbot, sources_output],
    ).then(lambda: "", None, msg_input)

    msg_input.submit(
        fn=respond,
        inputs=[msg_input, chatbot, use_rewrite, use_rerank],
        outputs=[chatbot, sources_output],
    ).then(lambda: "", None, msg_input)

demo.queue()  # Enables proper streaming updates
demo.launch()